# VietOCR GPU — Nôm Workspace
Luồng: phân loại/vùng mẫu trong ứng dụng → Xuất Colab → notebook này → Nhập kết quả → sửa nhãn.
Chọn Màu mực trong quyển ở bước Gắn nhãn trang trước khi xuất input; worker dùng profile đó để tìm dòng trên Colab.
Chọn Runtime → Change runtime type → GPU. Không dùng local runtime.
Không dùng EasyOCR. VietOCR ONNX và bộ tìm dòng được đóng gói cùng mã nguồn của repo.
Bộ tìm dòng vẫn chạy CPU; encoder/decoder nhận dạng dùng CUDA. Không bảo đảm box tự động đúng với nhiều cột.


In [ ]:
from google.colab import files, drive
drive.mount('/content/drive')
print('Chọn vietocr-colab-source.zip được xuất cùng phiên bản mã local')
uploaded = files.upload()


In [ ]:
import zipfile, pathlib, sys, subprocess
source = next(name for name in uploaded if name.endswith('.zip'))
ROOT = pathlib.Path('/content/nom-vietocr')
ROOT.mkdir(exist_ok=True)
with zipfile.ZipFile(source) as archive:
    for item in archive.infolist():
        target = (ROOT / item.filename).resolve()
        assert target.is_relative_to(ROOT.resolve()), 'Unsafe ZIP path'
    archive.extractall(ROOT)
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'onnxruntime', 'onnxruntime-gpu'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(ROOT / 'notebooks/requirements-colab.txt')], check=True)
# Chạy worker ở tiến trình mới để tránh xung đột thư viện đã import trong kernel.


In [ ]:
print('Chọn vietocr-colab-input.zip tải từ ứng dụng')
uploaded = files.upload()
BUNDLE = pathlib.Path(next(name for name in uploaded if name.endswith('.zip'))).resolve()
OUTPUT = pathlib.Path('/content/drive/MyDrive/NomWorkspace_VietOCR')
OUTPUT.mkdir(parents=True, exist_ok=True)
LIMIT = 3  # Thử 3 trang; đổi 0 để chạy toàn bộ. Cache theo gói/trang trên Drive.


In [ ]:
import os
env = {**os.environ, 'VI_OCR_DEVICE': 'cuda'}
subprocess.run([sys.executable, str(ROOT / 'backend/setup_vietocr.py')], cwd=ROOT, check=True)
# Import torch trước ONNX để nạp CUDA/cuDNN có sẵn trên Colab.
script = "import torch; assert torch.cuda.is_available(), 'Chưa có GPU Colab'; print(torch.cuda.get_device_name(0)); from backend.colab_worker import run; from pathlib import Path; print(run(Path(" + repr(str(BUNDLE)) + "), Path(" + repr(str(OUTPUT)) + "), Path(" + repr(str(ROOT)) + "), " + str(LIMIT) + "))"
subprocess.run([sys.executable, '-u', '-c', script], cwd=ROOT, env=env, check=True)
lock = subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True)
(OUTPUT / 'requirements-runtime-lock.txt').write_text(lock)


In [ ]:
import json
with zipfile.ZipFile(BUNDLE) as archive:
    manifest = json.loads(archive.read('manifest.json'))
result_zip = OUTPUT / (manifest['bundle_id'] + '-results.zip')
with zipfile.ZipFile(result_zip) as archive:
    errors = json.loads(archive.read('errors.json'))
    print('Lỗi:', errors)
    from IPython.display import display
    from PIL import Image
    import io
    for name in archive.namelist():
        if name.startswith('previews/'):
            image = Image.open(io.BytesIO(archive.read(name)))
            image.thumbnail((1000, 1400))
            display(image)
files.download(str(result_zip))


## Nhập lại ứng dụng
Dừng OCR local, chọn **Nhập kết quả Colab** ở đúng quyển sách. Gói thử 3 trang chỉ nhập 3 trang đó.
Để chạy hết, đổi LIMIT=0 rồi chạy worker và tải ZIP lại **trước khi nhập bản thử**.
Nếu đã nhập/sửa trang, hãy xuất gói mới: ứng dụng từ chối gói cũ khi phiên bản trang thay đổi.
Kết quả giữ trạng thái chưa duyệt, chỉ thay nháp Quốc ngữ; không sửa Nôm.
Kiểm tra box, chữ, thứ tự đọc trong giao diện hiện tại, rồi tiếp tục dóng hàng/xuất nhãn như trước.
Nếu không tìm được dòng, chọn vùng Quốc ngữ hẹp hơn theo từng cột ở ứng dụng rồi xuất lại.
